In [167]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import expr, col, round, avg, max, min, sum, count, when, lit
from itertools import product
from datetime import datetime
import pytz
import os
import logging

In [168]:

# Configura logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [169]:
def configure_spark():

    spark = (
        SparkSession.builder
        .appName("lakehouse-jupyter-churn")

        # conexão com cluster
        .master(os.getenv("SPARK_MASTER"))

        # 🚨 ESSENCIAL em Docker
        .config("spark.driver.host", os.getenv("SPARK_DRIVER_HOST"))
        .config("spark.driver.bindAddress", "0.0.0.0")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # S3 / MinIO
        .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

        .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
        .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))

        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
        )

        # performance
        .config("spark.sql.adaptive.enabled", "true")

        # Delta fix
        .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

In [170]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [171]:
#executar todas as datas abaixo em (  data_exec_inicial  ) 
#201808, 201807, 201806, 201805, 201804, 201803, 201802, 201801, 201712, 201711, 201710, 201709
# converte YYYYMM -> date
data_exec_inicial = 201808
# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int(
    (data_dt + relativedelta(months=2)).strftime("%Y%m")
)
data_exec_final

201810

In [172]:
spark = configure_spark()
spark

In [173]:
logging.info("Iniciando criação do book: pedidos")

df_orders = (
    spark.read.table("delta.`s3a://silver/olist_orders`")
)

df_orders.createOrReplaceTempView("df_orders")

2026-06-21 17:02:41,721 - INFO - Iniciando criação do book: pedidos


In [174]:
df_order_items = spark.read.table("delta.`s3a://silver/olist_order_items`")

df_order_items.createOrReplaceTempView("df_order_items")

In [175]:
df_orders_01 = spark.sql(f"""
WITH active_sellers AS (
    -- 1. Vendedores ativos na safra base
    SELECT DISTINCT
        oi.seller_id,
        o.safra AS safra_ref
    FROM df_orders o
    INNER JOIN df_order_items oi
        ON o.order_id = oi.order_id
    WHERE o.safra = {data_exec_inicial}
      AND o.order_status IN ('DELIVERED')
),

base AS (
    -- 2. Converte safra INT → DATE (padrão do seu projeto)
    SELECT
        *,
        TO_DATE(CONCAT(safra_ref, '01'), 'yyyyMMdd') AS data_ref
    FROM active_sellers
),

future_sales AS (
    -- 3. Vendas futuras no período
    SELECT DISTINCT
        oi.seller_id,
        o.safra AS safra_venda
    FROM df_orders o
    INNER JOIN df_order_items oi
        ON o.order_id = oi.order_id
    WHERE o.safra BETWEEN {data_exec_inicial} AND {data_exec_final}
      AND o.order_status IN ('DELIVERED')
),

future_base AS (
    -- 4. Converte safra futura para DATE
    SELECT
        *,
        TO_DATE(CONCAT(safra_venda, '01'), 'yyyyMMdd') AS data_venda
    FROM future_sales
),

sellers_with_future_sales AS (
    -- 5. Verifica vendas entre +1 e +2 meses após safra base
    SELECT DISTINCT
        a.seller_id,
        a.safra_ref
    FROM base a
    INNER JOIN future_base f
        ON a.seller_id = f.seller_id

    --WHERE f.data_venda BETWEEN
        --ADD_MONTHS(a.data_ref, 1)
        --AND ADD_MONTHS(a.data_ref, 2)

    WHERE f.data_venda = ADD_MONTHS(a.data_ref, 1)
       
)

-- 6. Target final de churn
SELECT
    a.seller_id,
    a.safra_ref AS safra,

    CASE
        WHEN f.seller_id IS NOT NULL THEN 0
        ELSE 1
    END AS churn_flag

FROM active_sellers a
LEFT JOIN sellers_with_future_sales f
    ON a.seller_id = f.seller_id
   AND a.safra_ref = f.safra_ref

ORDER BY
    a.seller_id,
    a.safra_ref;
""")

df_orders_01.createOrReplaceTempView("df_orders_01")

In [176]:
from pyspark.sql import functions as F

df_orders_01.groupBy("safra") \
    .agg(
        F.avg("churn_flag").alias("churn_rate"),
        F.count("*").alias("total_sellers")
    ) \
    .orderBy("safra") \
    .show(truncate=False)

+------+----------+-------------+
|safra |churn_rate|total_sellers|
+------+----------+-------------+
|201808|1.0       |1246         |
+------+----------+-------------+



In [177]:
spark.sql("""
-- 4) Calcula hash para idempotência (evita UPDATE sem mudança real)
CREATE OR REPLACE TEMP VIEW stage_orders_final AS
SELECT
  *,
  sha2(concat_ws('||',
    coalesce(seller_id,''),
    coalesce(cast(safra as int),''),
    coalesce(cast(churn_flag as int),'')
    --coalesce(product_id,''),
    --coalesce(seller_id,''),
    --coalesce(date_format(shipping_limit_date,'yyyy-MM-dd'),''),
    --coalesce(cast(price as string),''),
    --coalesce(cast(freight_value as string),'')
  ), 256) AS row_hash
FROM df_orders_01;
""")

DataFrame[]

In [178]:
spark.sql("""
        CREATE DATABASE IF NOT EXISTS gold
        LOCATION 's3a://gold'
""")

DataFrame[]

In [179]:
from delta.tables import DeltaTable

path = "s3a://gold/public_sellers_churn"

if not spark.catalog.tableExists("gold.public_sellers_churn"):
    
    # Se já existe Delta no storage → só registra
    if DeltaTable.isDeltaTable(spark, path):
        spark.sql(f"""
            CREATE TABLE gold.public_sellers_churn
            USING DELTA
            LOCATION '{path}'
        """)
    
    # Se não existe nada → cria do zero
    else:
        spark.sql(f"""
            CREATE TABLE gold.public_sellers_churn
            USING DELTA
            PARTITIONED BY (safra)
            LOCATION '{path}'
            AS SELECT * FROM stage_orders_final WHERE 1=0
        """)

In [ ]:
spark.sql("""
MERGE INTO gold.public_sellers_churn AS t
USING stage_orders_final AS s
ON t.seller_id = s.seller_id
AND t.safra = s.safra

WHEN MATCHED AND (t.row_hash IS NULL OR t.row_hash <> s.row_hash) THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
;
""")

In [ ]:
logging.info("Finalizando do publico e target")
spark.stop()